# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\hadia.0NYX\AppData\Local\Python\pythoncore-3.11-64\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

print(ds)

c:\Users\hadia.0NYX\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hadia.0NYX\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 104/104 [00:00<00:00, 3248.37 ex

Dataset({
    features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
    num_rows: 104
})


In [4]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("FlyRank/internship-warehouse")
print(configs)

['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']


In [5]:
from datasets import load_dataset

sample = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

first_row = next(iter(sample))

print(first_row)

{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}


In [11]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(march_file)

C:\Users\hadia.0NYX\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet


In [12]:
import duckdb

con = duckdb.connect()

march = con.sql(f"""
SELECT *
FROM read_parquet('{march_file}')
""").df()

print(march.shape)
march.head()

(9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [13]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet('{march_file}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,cnt


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [15]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet('{march_file}')
""").df()

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


## Unit of Analysis

**Unit of analysis**

One row represents the daily performance of a single content item for a single client on a specific report date.

The dataset grain is:

(report_date, client_hash_id, content_hash_id)

This was verified by grouping on these three fields and checking for duplicate combinations. The verification query returned zero duplicate rows, confirming that the grain is correct.

---

## Time Window

The analysed data covers **1 March 2026 to 31 March 2026**.

A total of **9,841,378 rows** are included in this month's partition.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

## Label
No prediction label is defined in this notebook. The notebook focuses on establishing the data contract.

## Context
- report_date
- client_hash_id
- content_hash_id
- gsc_data_available
- ga4_data_available

## Excluded
- client_hash_id (identifier only)
- content_hash_id (identifier only)
- Availability flags are used for filtering, not as model features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Query 1 – Verify the grain

The expected grain is one row per:

- report_date
- client_hash_id
- content_hash_id

The query below checks for duplicate combinations. An empty result confirms that the dataset grain is correct.

In [26]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet('{march_file}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""").df()

,report_date,client_hash_id,content_hash_id,cnt


## Query 2 – Verify the time window and row count

This query confirms the date range covered by the selected partition and the total number of rows.

In [27]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet('{march_file}')
""").df()

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


## Query 3 – Verify data availability

The assignment requires filtering using `IS TRUE`. The following queries count only the rows where Google Search Console and GA4 data are available.

In [28]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_after_filter
FROM read_parquet('{march_file}')
WHERE gsc_data_available IS TRUE
""").df()

,rows_after_filter
0,3611061


In [29]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_after_filter
FROM read_parquet('{march_file}')
WHERE ga4_data_available IS TRUE
""").df()

,rows_after_filter
0,413966


## Missing Value Analysis

This query measures the percentage of missing values for selected Google Search Console and GA4 fields in the March 2026 partition.

In [30]:
con.sql(f"""
SELECT
    ROUND(100.0 * AVG(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END),2) AS gsc_impressions_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END),2) AS gsc_clicks_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END),2) AS gsc_position_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END),2) AS ga4_pageviews_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END),2) AS ga4_sessions_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN scroll_events IS NULL THEN 1 ELSE 0 END),2) AS scroll_events_missing_pct
FROM read_parquet('{march_file}')
""").df()

,gsc_impressions_missing_pct,gsc_clicks_missing_pct,gsc_position_missing_pct,ga4_pageviews_missing_pct,ga4_sessions_missing_pct,scroll_events_missing_pct
0,0.0,0.0,63.31,30.67,30.67,30.67


### Interpretation

The verification queries confirm that:

- The dataset grain is one row per **report_date**, **client_hash_id**, and **content_hash_id**, with no duplicate records.
- The selected partition covers **1 March 2026 to 31 March 2026** and contains **9,841,378 rows**.
- Filtering with `gsc_data_available IS TRUE` leaves **3,611,061** usable rows.
- Filtering with `ga4_data_available IS TRUE` leaves **413,966** usable rows.
- Google Search Console impressions and clicks contain no missing values, while `gsc_avg_position` and several GA4 metrics contain missing values. These missing values reflect data availability and should be considered before modelling.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This analysis uses only the March 2026 partition of the warehouse and therefore does not represent the full history available in the dataset.

Not every client has complete Google Search Console or GA4 data. The availability flags show that some rows contain unavailable data rather than true zero values, so filtering is required before analysis.

The dataset contains observational performance metrics only. It cannot explain why content performed well or poorly, nor can it measure factors such as content quality, algorithm updates, or marketing decisions.

## Five-Feature Frame

The following feature frame contains five candidate features that are available before the prediction decision and can be considered for future modelling.

In [31]:
feature_frame = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet('{march_file}')
WHERE gsc_data_available IS TRUE
LIMIT 10
""").df()

feature_frame

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>
5,239,1,7.347280,<NA>,<NA>
6,191,0,7.832461,<NA>,<NA>
7,55,0,3.272727,<NA>,<NA>
8,77,0,5.636364,<NA>,<NA>
9,2,0,4.500000,<NA>,<NA>


### Feature availability

- **gsc_impressions** – Knowable at the decision moment because Search Console has already recorded the day's impressions.
- **gsc_clicks** – Knowable at the decision moment because clicks are observed before making future predictions.
- **gsc_avg_position** – Knowable because it is calculated from historical Search Console performance.
- **ga4_pageviews** – Knowable only when GA4 data is available for the client and date.
- **ga4_sessions** – Knowable only when GA4 tracking is available for the client and date.

## Leakage Demonstration

A feature derived directly from the prediction target (or future information) would introduce data leakage and produce unrealistically high model performance.

For example, if a future performance metric or a label-derived column were included as a feature, the model could appear nearly perfect during evaluation while failing on unseen data.

The leakage feature should therefore be removed, and only features available at the prediction decision time should be retained.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.